In [ ]:
import os
# —— 1. 环境变量 & 路径 ——
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from unsloth import FastLanguageModel
import json, torch
from datasets import load_dataset, concatenate_datasets
from transformers import Trainer, TrainingArguments

MODEL_DIR = "" # where to save
DATA_PATH = "" # where to load training data
MAX_LEN = 8192

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/yang3j7/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# —— 2. 加载原始数据集 ——
all_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = all_dataset
print("截取后的数据量:", len(dataset))

# —— 3. 加载模型 & tokenizer ——
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-14B",  # different base model to finetune
    max_seq_length = MAX_LEN,                           # input length
    dtype          = torch.bfloat16,
    load_in_4bit   = False,
)
model.train()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


截取后的数据量: 2328


==((====))==  Unsloth 2025.8.5: Fast Qwen3 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 6/6 [00:05<00:00,  1.19it/s]


In [4]:
# %%
# —— 4. 定义 system + prompt template ——
system_msg = {
    "role": "system",
    "content": "You are a helpful assistant."
}

prompt_tpl = """### Instruction:
Please determine whether there is a non-crash functional bug in the following android app UI interaction trace.

Think step by step inside <think>...</think>, then give your final JSON answer.

### Input:
{actions}

### Output format example:
<think>
your detailed reasoning here...
</think>

{{
  "is_bug": "Yes" or "No",
  "reason": "brief explanation"
}}
"""


In [5]:
# %%
import json

def _norm_is_bug(x):
    # 统一 "Yes"/"No"
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"yes", "true", "1"}:
            return "Yes"
        if s in {"no", "false", "0"}:
            return "No"
        return "No"
    if isinstance(x, (bool, int)):
        return "Yes" if bool(x) else "No"
    return "No"

def normalize_reasoning(value, pretty=False, max_chars=None):
    """统一处理 reasoning 字段: str / list / dict 都转成字符串"""
    def _to_str(x):
        if isinstance(x, dict):
            return json.dumps(x, ensure_ascii=False, sort_keys=True, indent=2 if pretty else None)
        return str(x)

    if isinstance(value, list):
        s = " ".join(_to_str(x).strip() for x in value)
    elif isinstance(value, dict):
        s = _to_str(value)
    else:
        s = _to_str(value)

    s = s.strip()
    if max_chars and len(s) > max_chars:
        s = s[:max_chars] + "…"
    return s

def format_example(example):
    # 1) 读取训练样本
    trace   = example.get("gen_trace", "")
    gold_rs = normalize_reasoning(example.get("thoughts", ""), pretty=False)

    j       = example.get("judgement", {}) or {}
    is_bug  = _norm_is_bug(j.get("is_bug", "No"))
    reason  = (j.get("reason") or "").strip()

    # 2) 输入只包含 actions
    user_msg = {
        "role": "user",
        "content": prompt_tpl.format(actions=trace),
    }
    prompt_text = tokenizer.apply_chat_template(
        [system_msg, user_msg],
        tokenize=False,
        add_generation_prompt=True,
    )

    # 3) 输出包含 reasoning + JSON
    out_obj = {"is_bug": is_bug, "reason": reason}
    output_text = (
        "<think>\n" + gold_rs + "\n</think>\n\n" +
        json.dumps(out_obj, ensure_ascii=False) +
        tokenizer.eos_token
    )
    # 4) 拼接 & tokenize（只对 assistant 段算损失）
    full_text   = prompt_text + output_text
    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    prompt_tokens = tokenizer(prompt_text, add_special_tokens=False)

    input_ids  = full_tokens["input_ids"]
    prompt_len = len(prompt_tokens["input_ids"])
    labels     = [-100] * prompt_len + input_ids[prompt_len:]

    if len(labels) != len(input_ids) or all(l == -100 for l in labels):
        return {}

    full_tokens["labels"] = labels
    return full_tokens

# 5.7 重新 map
dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
)


Map:   1%|          | 19/2328 [00:00<00:13, 175.89 examples/s]

Map: 100%|██████████| 2328/2328 [00:15<00:00, 154.34 examples/s]


In [6]:
def data_collator(features):
    # 动态找本 batch 的最大长度
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id  = tokenizer.pad_token_id
    batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
    }
    for f in features:
        ids    = f["input_ids"]
        mask   = f["attention_mask"]
        labels = f["labels"]

        pad_len = max_len - len(ids)
        batch["input_ids"].append(ids + [pad_id] * pad_len)
        batch["attention_mask"].append(mask + [0] * pad_len)
        batch["labels"].append(labels + [-100] * pad_len)   # 只给 labels 补 -100

    # 转成张量
    return {
        "input_ids": torch.tensor(batch["input_ids"], dtype=torch.long),
        "attention_mask": torch.tensor(batch["attention_mask"], dtype=torch.long),
        "labels": torch.tensor(batch["labels"], dtype=torch.long),
    }


In [7]:
# —— 6. 应用 LoRA（peft） —— 
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha         = 32,
    lora_dropout       = 0.05,
    bias               = "none",
    use_gradient_checkpointing = True,
    random_state       = 42,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth 2025.8.5 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [8]:
# —— 7. TrainingArguments：batch=1 —— 
# MODEL_DIR = "/home/yang3j7/finetune/test0717/new_trained_model_prompt_with_chat_template_with_output_format"

training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
)

In [9]:
# —— 8. 初始化 Trainer ——
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset,
    data_collator    = data_collator,   # ← 新增
    processing_class = tokenizer,
)

In [10]:
# 9. 开始训练 and save 
trainer.train()
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,328 | Num Epochs = 3 | Total steps = 6,984
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 20,971,520 of 14,789,278,720 (0.14% trained)


Step,Training Loss
20,1.829900
40,1.859400
60,1.687700
80,1.497600
100,1.404300
120,1.384600
140,1.320000
160,1.337700
180,1.271800
200,1.211200


Unsloth: Will smartly offload gradients to save VRAM!


('/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/tokenizer_config.json',
 '/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/special_tokens_map.json',
 '/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/chat_template.jinja',
 '/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/vocab.json',
 '/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/merges.txt',
 '/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/added_tokens.json',
 '/home/yang3j7/NCF0729/test0923/sft_trained_model_thinking_summary_by_gemini_shuffled_batch1_qwen3_2/tokenizer.json')

In [11]:
# --- 清理训练期对象，减少显存碎片对后续推理的影响 ---
del trainer
del dataset
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()